# Lab Day 10: Qwen3-VL-4B — định vị hư hại mặt đường
Notebook hoàn chỉnh E0–E4 theo `GUIDE.md`. Chạy từ trên xuống trên **Colab GPU T4 16 GB**.
E1–E3 dùng 300 optimizer steps mỗi lần; E4 chọn ablation kích thước dữ liệu 10%/100%.
Các cell chỉ định nghĩa hàm sẽ chưa huấn luyện cho tới phần 7. Không sửa công cụ chấm của đề.
Kết quả F1 và báo cáo phải lấy từ lần chạy thật; notebook không điền số liệu giả.

## 0. Môi trường (Colab T4)

In [ ]:
from pathlib import Path
import os, sys, subprocess

# Chạy được cả khi notebook đang ở repo local và khi mở riêng bằng Colab.
REPO_URL = "https://github.com/Le-Anh-Duy/K4-Track-4-Day10-VisionLanguageModel.git"
if not Path("requirements-colab.txt").is_file():
    checkout = Path("/content/K4-Track-4-Day10-VisionLanguageModel")
    if not checkout.exists():
        subprocess.run(["git", "clone", REPO_URL, str(checkout)], check=True)
    os.chdir(checkout)
ROOT = Path.cwd()
print("Repo:", ROOT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                "requirements-colab.txt", "matplotlib", "pytest", "tqdm"], check=True)
# Nếu đã import torch/transformers trước khi cài: restart runtime rồi chạy lại.

In [ ]:
import gc, json, math, random, re, shutil, time, tempfile, traceback
import torch
import yaml
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from tqdm.auto import tqdm
from scripts.labcommon import CLASSES, read_jsonl

print("torch:", torch.__version__, "CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Cần CUDA GPU: Colab > Runtime > Change runtime type > T4 GPU.")
print(torch.cuda.get_device_name(0))
def alert(message, **details):
    print(f"[ALERT] {message}", details if details else "", flush=True)
    try:
        log = ROOT / "work/alerts.jsonl"
        log.parent.mkdir(parents=True, exist_ok=True)
        with log.open("a", encoding="utf-8") as handle:
            handle.write(json.dumps(dict(time=time.strftime("%Y-%m-%d %H:%M:%S"),
                         message=message, **details), ensure_ascii=False, default=str)+"\n")
    except OSError as exc:
        print(f"[ALERT] Không ghi được log: {exc}", flush=True)

def safe_task(label, fn):
    try:
        return fn()
    except Exception as exc:
        alert(f"{label}: THẤT BẠI, tiếp tục phần độc lập kế tiếp", error=str(exc),
              traceback=traceback.format_exc())
        return None
    finally:
        gc.collect()
        torch.cuda.empty_cache()

test_result = subprocess.run([sys.executable, "-m", "pytest", "-q"])
if test_result.returncode:
    alert("Bộ test repo chưa qua; xem output phía trên", returncode=test_result.returncode)

# Bật để lưu checkpoint thẳng lên Drive; không bắt buộc tài khoản Drive khi chạy local.
USE_DRIVE = True
try:
    from google.colab import drive
except ImportError:
    drive = None
# Checkpoint chính ở ổ local; Drive chỉ là bản sao, lỗi Drive không ngắt train.
RUN_ROOT = ROOT / "work/runs"
DRIVE_RUN_ROOT = None
if USE_DRIVE and drive is not None:
    try:
        drive.mount("/content/drive")
        DRIVE_RUN_ROOT = Path("/content/drive/MyDrive/lab_day10/runs")
        DRIVE_RUN_ROOT.mkdir(parents=True, exist_ok=True)
    except Exception as exc:
        alert("Không mount/ghi được Drive; chỉ lưu local (mất khi Colab xoá máy ảo)", error=str(exc))
        DRIVE_RUN_ROOT = None
RUN_ROOT.mkdir(parents=True, exist_ok=True)
DATA_ROOT = ROOT / "data"

## 1. Dữ liệu: tải và chia split
Mặc định tải Kaggle. Có thể đổi `DATA_SOURCE` sang thư mục đã giải nén hoặc file zip.
Secrets chỉ được đọc khi cần tải; không ghi khóa vào notebook.

In [ ]:
DATA_SOURCE = "kaggle"  # hoặc "/content/dataset.zip", "/content/road-damage"
if not all((DATA_ROOT / "splits" / f"{s}.jsonl").is_file() for s in ("train", "val", "test")):
    args = [sys.executable, "scripts/prepare_data.py"]
    if DATA_SOURCE == "kaggle":
        if not (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")):
            try:
                from google.colab import userdata
                for key in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
                    os.environ[key] = userdata.get(key)
            except Exception as exc:
                raise RuntimeError("Cấu hình Colab Secrets KAGGLE_USERNAME/KAGGLE_KEY hoặc đổi DATA_SOURCE.") from exc
        args += ["--kaggle"]
    else:
        args += ["--zip" if DATA_SOURCE.lower().endswith(".zip") else "--source", DATA_SOURCE]
    subprocess.run(args, check=True)

splits = {s: read_jsonl(DATA_ROOT / "splits" / f"{s}.jsonl") for s in ("train", "val", "test")}
for s, rows in splits.items():
    assert rows, f"Split rỗng: {s}"
    counts = {c: sum(o["label"] == c for r in rows for o in r["objects"]) for c in CLASSES}
    print(s, len(rows), "images; boxes:", counts)
sessions = {s: {r["session"] for r in rows} for s, rows in splits.items()}
assert not (sessions["train"] & sessions["val"] or sessions["train"] & sessions["test"] or sessions["val"] & sessions["test"])
train_samples, val_samples, test_samples = (splits[s] for s in ("train", "val", "test"))

def open_image(sample):
    with Image.open(DATA_ROOT / sample["image_path"]) as image:
        return image.convert("RGB")  # giữ kích thước gốc, không thumbnail 512px

def draw_sample(sample, boxes=None):
    image = open_image(sample)
    draw = ImageDraw.Draw(image)
    colors = {"pothole": "red", "crack": "yellow", "manhole": "cyan"}
    if boxes is None:
        boxes = []
        for obj in sample["objects"]:
            xc, yc, w, h = obj["yolo"]
            boxes.append({"label": obj["label"], "bbox_2d":
                          [(xc-w/2)*1000, (yc-h/2)*1000, (xc+w/2)*1000, (yc+h/2)*1000]})
    for obj in boxes:
        x1, y1, x2, y2 = obj["bbox_2d"]
        xy = [x1*image.width/1000, y1*image.height/1000, x2*image.width/1000, y2*image.height/1000]
        draw.rectangle(xy, outline=colors[obj["label"]], width=2)
        draw.text((xy[0], max(0, xy[1]-12)), obj["label"], fill=colors[obj["label"]])
    return image

def preview_samples():
    fig, axes = plt.subplots(2, 3, figsize=(18, 7))
    for ax, sample in zip(axes.flat, train_samples[:6]):
        ax.imshow(draw_sample(sample))
        ax.set_title(f"{sample['image_id']} ({len(sample['objects'])} boxes)", fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

safe_task("Vẽ ảnh minh hoạ", preview_samples)

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
def yolo_to_box(label, yolo):
    if label not in CLASSES or len(yolo) != 4:
        raise ValueError("Nhãn/box không hợp lệ")
    if not all(math.isfinite(v) and 0 <= v <= 1 for v in yolo):
        raise ValueError("YOLO phải hữu hạn trong [0, 1]")
    xc, yc, w, h = yolo
    if w == 0 or h == 0:
        return None  # Nhãn suy biến được loader của đề chấp nhận; bỏ khỏi target SFT.
    xy = [max(0, min(1000, round(v*1000))) for v in (xc-w/2, yc-h/2, xc+w/2, yc+h/2)]
    # Giữ box ít nhất 1 đơn vị kể cả khi làm tròn tại biên 1000.
    for lo, hi in ((0, 2), (1, 3)):
        if xy[hi] <= xy[lo]:
            xy[lo] = min(xy[lo], 999)
            xy[hi] = xy[lo] + 1
    return {"label": label, "bbox_2d": xy}

def objects_to_answer(objects):
    boxes = []
    for obj in objects:
        box = yolo_to_box(obj["label"], obj["yolo"])
        if box is not None:
            boxes.append(box)
    boxes.sort(key=lambda b: (b["bbox_2d"][1], b["bbox_2d"][0]))
    return json.dumps(boxes, separators=(",", ":"))

PROMPT = ('Locate road damage: pothole, crack, manhole. Return ONLY a JSON list '
          'of {"label":"pothole|crack|manhole","bbox_2d":[x1,y1,x2,y2]}. '
          'Coordinates are integers normalized to 0-1000 relative to image width/height. '
          'Use top-left and bottom-right corners. If no objects, return [].')
assert objects_to_answer([]) == "[]"
assert objects_to_answer([{"label": "crack", "yolo": [0.5, 0.5, 0, 0.2]}]) == "[]"
assert yolo_to_box("crack", [0.5, 0.5, 0.2, 0.4])["bbox_2d"] == [400, 300, 600, 700]
assert yolo_to_box("crack", [1, 1, 0.0001, 0.0001])["bbox_2d"] == [999, 999, 1000, 1000]
def prepare_training_rows(rows):
    valid, zero_area, dropped = [], 0, 0
    for row in rows:
        try:
            answer = objects_to_answer(row["objects"])
            with Image.open(DATA_ROOT / row["image_path"]) as image:
                image.verify()
            zero_area += len(row["objects"]) - len(json.loads(answer))
            valid.append(row)
        except (OSError, ValueError, KeyError, TypeError, OverflowError) as exc:
            dropped += 1
            alert("Bỏ ảnh train hỏng/nhãn không hợp lệ", image_id=row.get("image_id"), error=str(exc))
    if not valid:
        raise ValueError("Không còn ảnh train hợp lệ; cần sửa dữ liệu trước khi train.")
    if zero_area or dropped:
        alert("Kiểm tra dữ liệu train hoàn tất; không sửa GT/split của bộ chấm",
              zero_area_boxes=zero_area, dropped_train_images=dropped, usable_images=len(valid))
    return valid

train_samples = prepare_training_rows(train_samples)
sample = train_samples[0]
answer = objects_to_answer(sample["objects"])
for box in json.loads(answer):
    assert all(0 <= v <= 1000 for v in box["bbox_2d"])
print(answer)
def preview_roundtrip():
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for ax, image, title in zip(axes, [draw_sample(sample), draw_sample(sample, json.loads(answer))], ["YOLO gốc", "JSON → pixel"]):
        ax.imshow(image); ax.set_title(title); ax.axis("off")
    plt.show()

safe_task("Vẽ kiểm tra toạ độ", preview_roundtrip)

## 3. E0: zero-shot
NF4 chỉ cho LLM, giữ vision/projector và lm_head fp16; đặt toàn model trên GPU 0.
Giữ ảnh gốc, dùng chung processor/prompt cho mọi thí nghiệm. Output đúng `{image_id, raw}`.

In [ ]:
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration, set_seed

MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"
processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side = "left"

def load_base_model():
    quantization = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True, llm_int8_skip_modules=["visual", "lm_head"])
    return Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID, quantization_config=quantization, dtype=torch.float16,
        device_map={"": 0}, attn_implementation="sdpa")

def user_message(image):
    return {"role": "user", "content": [{"type": "image", "image": image},
                                          {"type": "text", "text": PROMPT}]}

def predict(model, rows, path, batch_size=4, max_new_tokens=256):
    model.eval()
    model.config.use_cache = True
    processor.tokenizer.padding_side = "left"
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    def infer_chunk(chunk):
        inputs = generated = None
        failure = None
        try:
            chats = [[user_message(open_image(row))] for row in chunk]
            inputs = processor.apply_chat_template(chats, tokenize=True, add_generation_prompt=True,
                       return_dict=True, return_tensors="pt", padding=True)
            inputs.pop("token_type_ids", None)
            inputs = inputs.to(model.device)
            with torch.autocast("cuda", dtype=torch.float16):
                generated = model.generate(**inputs, do_sample=False, max_new_tokens=max_new_tokens)
            return processor.batch_decode(generated[:, inputs["input_ids"].shape[1]:],
                       skip_special_tokens=True, clean_up_tokenization_spaces=False)
        except (torch.cuda.OutOfMemoryError, OSError) as exc:
            failure = str(exc)
        finally:
            del inputs, generated
        # Chỉ thử batch nhỏ hơn sau khi giải phóng tensor của lần lỗi.
        gc.collect()
        torch.cuda.empty_cache()
        if len(chunk) > 1:
            alert("Inference lỗi; chia đôi batch và thử lại", batch_size=len(chunk), error=failure)
            middle = len(chunk)//2
            return infer_chunk(chunk[:middle]) + infer_chunk(chunk[middle:])
        alert("Không dự đoán được ảnh; ghi raw rỗng để bộ chấm tính output không hợp lệ",
              image_id=chunk[0]["image_id"], error=failure)
        return [""]

    temporary = path.with_suffix(".jsonl.tmp")
    with temporary.open("w", encoding="utf-8") as handle, torch.inference_mode():
        for start in tqdm(range(0, len(rows), batch_size), desc=path.stem):
            chunk = rows[start:start+batch_size]
            raw = infer_chunk(chunk)
            for row, text in zip(chunk, raw):
                handle.write(json.dumps({"image_id": row["image_id"], "raw": text}, ensure_ascii=False)+"\n")
            handle.flush()
    temporary.replace(path)
    return path

def score_run(pred_path, name, split="test", meta=None, evidence_name=None):
    out = ROOT / "report/evidence" / f"{evidence_name or name.lower()}.json"
    cmd = [sys.executable, "scripts/score.py", "--pred", str(pred_path), "--name", name,
           "--split", split, "--out", str(out)]
    for key, value in (meta or {}).items():
        cmd += ["--meta", f"{key}={value}"]
    subprocess.run(cmd, check=True)
    result = json.loads(out.read_text(encoding="utf-8"))
    if result.get("parse_valid_rate", 0) < 0.95 or result.get("missing_predictions", 0):
        alert("Output có JSON không hợp lệ/thiếu; điểm vẫn được giữ nguyên", run=name, split=split,
              parse_valid_rate=result.get("parse_valid_rate"), missing=result.get("missing_predictions"))
    return result

def run_e0():
    model = None
    try:
        model = load_base_model()
        predict(model, test_samples, ROOT / "work/e0.jsonl")
        result = safe_task("Chấm E0", lambda: score_run(ROOT / "work/e0.jsonl", "E0"))
        for row in read_jsonl(ROOT / "work/e0.jsonl")[:5]:
            print(row)
        return result
    finally:
        del model

e0 = safe_task("E0 zero-shot", run_e0)

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss
Encode prompt và hội thoại đầy đủ bằng **cùng processor và ảnh**; xác nhận prefix trước khi mask.
Chỉ tính loss trên JSON và token kết thúc. Pad các tensor chuỗi bên phải; nối patch ảnh/grid
theo chiều 0, không pad patch ảnh. Không cắt chuỗi vì có thể mất token ảnh hoặc JSON.

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

def encode_example(image, answer):
    user = user_message(image)
    kwargs = dict(tokenize=True, return_dict=True, return_tensors="pt")
    prompt = processor.apply_chat_template([user], add_generation_prompt=True, **kwargs)
    full = processor.apply_chat_template([user, {"role": "assistant", "content": answer}],
                                          add_generation_prompt=False, **kwargs)
    full.pop("token_type_ids", None)
    n = prompt["input_ids"].shape[1]
    if not torch.equal(full["input_ids"][0, :n], prompt["input_ids"][0]):
        raise ValueError("Chat template thay đổi prefix: không được mask bằng độ dài ước lượng.")
    labels = full["input_ids"].clone()
    labels[:, :n] = -100
    labels[full["attention_mask"] == 0] = -100
    assert (labels != -100).any(), "Không còn token đáp án"
    full["labels"] = labels
    return dict(full)

def collate(batch):
    batch = [item for item in batch if item is not None]
    if not batch:
        return None
    result = {}
    sequence_keys = {"input_ids", "attention_mask", "labels", "mm_token_type_ids"}
    if any(set(item) != set(batch[0]) for item in batch):
        raise ValueError("Batch có bộ trường không đồng nhất")
    for key in batch[0]:
        values = [item[key] for item in batch]
        if key in sequence_keys:
            pad = -100 if key == "labels" else (processor.tokenizer.pad_token_id if key == "input_ids" else 0)
            result[key] = pad_sequence([v.squeeze(0) for v in values], batch_first=True, padding_value=pad)
        elif key in {"pixel_values", "image_grid_thw"}:
            result[key] = torch.cat(values, dim=0)
        else:
            raise ValueError(f"Trường processor chưa xử lý: {key}")
    return result

class RoadDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, index):
        row = self.rows[index]
        try:
            image = open_image(row)
            answer = objects_to_answer(row["objects"])
        except (OSError, ValueError, KeyError, TypeError) as exc:
            alert("Bỏ mẫu train không đọc được", image_id=row.get("image_id"), error=str(exc))
            return None
        # Prefix/mask sai là lỗi hệ thống: để run báo thất bại, không âm thầm bỏ cả dataset.
        return encode_example(image, answer)

encoded = RoadDataset(train_samples)[0]
supervised = encoded["labels"][0] != -100
decoded = processor.tokenizer.decode(encoded["input_ids"][0][supervised], skip_special_tokens=False)
expected = objects_to_answer(train_samples[0]["objects"])
assert decoded.startswith(expected) and "<|im_end|>" in decoded[len(expected):]
image_id = processor.tokenizer.convert_tokens_to_ids("<|image_pad|>")
image_tokens = int((encoded["input_ids"] == image_id).sum())
grid_tokens = int(encoded["image_grid_thw"].prod(-1).sum()) // processor.image_processor.merge_size**2
assert image_tokens == grid_tokens
print("Vision tokens:", image_tokens, "(640×360 mặc định của đề: 220)")
print("Chỉ các token này có loss:", decoded)
assert (encoded["labels"][encoded["input_ids"] == image_id] == -100).all()
batch_check = collate([encoded, encode_example(open_image(train_samples[1]), "[]")])
assert (batch_check["labels"][batch_check["attention_mask"] == 0] == -100).all()
assert batch_check["image_grid_thw"].shape[0] == 2
assert batch_check["pixel_values"].shape[0] == sum(int(g.prod()) for g in batch_check["image_grid_thw"])
del encoded, batch_check
# E0 đã được giải phóng trong run_e0().
gc.collect()
torch.cuda.empty_cache()

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
from peft import LoraConfig, get_peft_model, PeftModel

LLM_TARGET = r"model\.language_model\.layers\.\d+\.(?:self_attn\.(?:q|k|v|o)_proj|mlp\.(?:gate|up|down)_proj)"
VIT_TARGET = r"model\.visual\.blocks\.\d+\.(?:attn\.(?:qkv|proj)|mlp\.(?:linear_fc1|linear_fc2))"

def is_projector(name):
    return "visual.merger." in name or "visual.deepstack_merger_list." in name

def apply_setup(base, cfg):
    base.requires_grad_(False)
    regex = LLM_TARGET if cfg["lora"]["targets"] == "llm" else f"(?:{LLM_TARGET}|{VIT_TARGET})"
    matched = [n for n, _ in base.named_modules() if re.fullmatch(regex, n)]
    expected = 252 if cfg["lora"]["targets"] == "llm" else 348
    if len(matched) != expected:
        raise ValueError(f"Target LoRA: {len(matched)} != {expected}; kiểm tra named_modules/model version.")
    # Không dùng prepare_model_for_kbit_training: nó cast toàn bộ ViT sang fp32,
    # tăng VRAM ngoài yêu cầu; chỉ projector cần fp32 cho GradScaler.
    tuned = get_peft_model(base, LoraConfig(
        r=cfg["lora"]["r"], lora_alpha=cfg["lora"]["alpha"],
        lora_dropout=cfg["lora"]["dropout"], target_modules=regex, bias="none"))
    for name, param in tuned.named_parameters():
        if is_projector(name) and cfg["train_projector"]:
            param.data = param.data.float()
            param.requires_grad_(True)
        elif param.requires_grad:
            param.data = param.data.float()  # fp16 gradients không unscale được
    tuned.enable_input_require_grads()
    tuned.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    tuned.config.use_cache = False
    if hasattr(tuned.config, "text_config"):
        tuned.config.text_config.use_cache = False
    assert all("lora_" in n or (cfg["train_projector"] and is_projector(n))
               for n, p in tuned.named_parameters() if p.requires_grad)
    assert all(p.dtype == torch.float32 for p in tuned.parameters() if p.requires_grad)
    tuned.print_trainable_parameters()
    return tuned

## 6. Vòng huấn luyện
Batch mặc định 1, tích luỹ 8 (effective batch 8). Chỉ đếm optimizer steps thành công.
Loss/gradient NaN hoặc OOM forward/backward: xoá toàn bộ gradient của lần thử và tiếp tục;
20 lần lỗi liên tiếp hoặc 100 lần tổng cộng sẽ dừng riêng run đó, ghi log và chuyển setup khác.
Checkpoint ở step 10, mỗi 50 steps và cuối run: lưu local trước, backup Drive sau.
Giữ hai checkpoint local gần nhất; lỗi backup/cleanup chỉ cảnh báo.
Checkpoint chưa chứa optimizer/RNG nên run ngắt giữa chừng vẫn train lại từ đầu.

In [ ]:
from transformers import get_cosine_schedule_with_warmup

def write_json_atomic(path, value):
    path = Path(path)
    temporary = path.with_suffix(".json.tmp")
    temporary.write_text(json.dumps(value, indent=2), encoding="utf-8")
    temporary.replace(path)

def remove_checkpoint_dir(folder, target):
    folder, target = Path(folder).resolve(), Path(target).resolve()
    if target.parent != folder or not target.name.startswith(("checkpoint-", ".saving-")):
        raise ValueError(f"Không xoá đường dẫn ngoài thư mục checkpoint: {target}")
    shutil.rmtree(target)

def prune_checkpoints(folder, current):
    try:
        previous = sorted((p for p in folder.glob("checkpoint-*") if p != current),
                          key=lambda p: p.stat().st_mtime, reverse=True)
        for stale in previous[1:]:
            remove_checkpoint_dir(folder, stale)
    except (OSError, ValueError) as exc:
        alert("Không dọn được checkpoint cũ; bản mới vẫn được giữ", folder=str(folder), error=str(exc))

def save_checkpoint(model, folder, cfg, stats):
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    # Lưu phiên bản mới riêng; chỉ đổi con trỏ sau khi toàn bộ trọng số đã lưu xong.
    staging = Path(tempfile.mkdtemp(prefix=".saving-", dir=folder))
    try:
        model.save_pretrained(staging)
        processor.save_pretrained(staging)
        extra = {n: p.detach().cpu().clone() for n, p in model.named_parameters()
                 if p.requires_grad and is_projector(n)}
        if cfg["train_projector"]:
            if not extra:
                raise ValueError("Thiếu projector cần lưu")
            torch.save(extra, staging / "extra_trainable.pt")
        state = {**stats, "config": cfg}
        write_json_atomic(staging / "run.json", state)
        target = folder / f"checkpoint-{stats['steps']:06d}-{staging.name[8:]}"
        staging.rename(target)
        state["checkpoint"] = target.name
        write_json_atomic(folder / "run.json", state)
    finally:
        if staging.exists():
            try:
                remove_checkpoint_dir(folder, staging)
            except (OSError, ValueError) as exc:
                alert("Không dọn được file checkpoint tạm", folder=str(staging), error=str(exc))
    # Backup Drive chỉ sau khi checkpoint local hoàn chỉnh.
    if DRIVE_RUN_ROOT is not None:
        try:
            backup = DRIVE_RUN_ROOT / folder.name
            shutil.copytree(target, backup / target.name, dirs_exist_ok=True)
            write_json_atomic(backup / "run.json", state)
            prune_checkpoints(backup, backup / target.name)
        except Exception as exc:
            alert("Backup Drive thất bại; checkpoint local vẫn dùng được", folder=str(folder), error=str(exc))
    # Luôn giữ bản hiện tại, kể cả khi train lại có số step thấp hơn run cũ.
    prune_checkpoints(folder, target)

def checkpoint_safely(model, folder, cfg, stats, attempts=1):
    for attempt in range(attempts):
        try:
            save_checkpoint(model, folder, cfg, stats)
            return True
        except (OSError, RuntimeError) as exc:
            alert("Lưu checkpoint thất bại", run=cfg["name"], step=stats["steps"], error=str(exc))
            if attempt+1 < attempts:
                time.sleep(1)
    return False

def train_run(cfg, rows, folder):
    model = optimizer = scheduler = scaler = loader = iterator = batch = loss = None
    params = []
    try:
        set_seed(cfg["seed"])
        model = apply_setup(load_base_model(), cfg)
        params = [p for p in model.parameters() if p.requires_grad]
        count = sum(p.numel() for p in params)
        loader = DataLoader(RoadDataset(rows), batch_size=cfg["batch_size"], shuffle=True,
                            collate_fn=collate, num_workers=0)
        iterator = iter(loader)
        optimizer = torch.optim.AdamW(params, lr=float(cfg["lr"]), weight_decay=0)
        scheduler = get_cosine_schedule_with_warmup(optimizer,
            num_warmup_steps=max(1, math.ceil(cfg["max_steps"]*0.05)), num_training_steps=cfg["max_steps"])
        scaler = torch.amp.GradScaler("cuda", init_scale=128.0)
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
        start = time.perf_counter()
        losses, step, rejected, consecutive_bad = [], 0, 0, 0
        model.train()
        with tqdm(total=cfg["max_steps"], desc=cfg["name"]) as progress:
            while step < cfg["max_steps"]:
                optimizer.zero_grad(set_to_none=True)
                loss_sum, problem = 0.0, None
                try:
                    for _ in range(cfg["grad_accum"]):
                        try:
                            batch = next(iterator)
                        except StopIteration:
                            iterator = iter(loader)
                            batch = next(iterator)
                        if batch is None:
                            problem = "Batch không còn mẫu đọc được"
                            break
                        batch = {k: v.to(model.device) for k, v in batch.items()}
                        with torch.autocast("cuda", dtype=torch.float16):
                            loss = model(**batch).loss
                        if not torch.isfinite(loss).item():
                            problem = "Loss NaN/Inf; bỏ toàn bộ tích luỹ của lần thử này"
                            break
                        loss_sum += loss.detach().item()/cfg["grad_accum"]
                        scaler.scale(loss/cfg["grad_accum"]).backward()
                        loss = batch = None
                except torch.cuda.OutOfMemoryError:
                    problem = "OOM forward/backward; bỏ lần thử này, không cập nhật trọng số"
                if problem is None:
                    scaler.unscale_(optimizer)
                    norm = torch.nn.utils.clip_grad_norm_(params, 1.0)
                    if not torch.isfinite(norm).item():
                        problem = "Gradient NaN/Inf; giảm AMP scale và bỏ lần cập nhật"
                        scaler.update(new_scale=max(scaler.get_scale()/2, 1e-8))
                if problem is not None:
                    optimizer.zero_grad(set_to_none=True)
                    loss = batch = None
                    gc.collect()
                    torch.cuda.empty_cache()
                    rejected += 1
                    consecutive_bad += 1
                    alert(problem, run=cfg["name"], completed_steps=step, rejected_updates=rejected)
                    if consecutive_bad >= 20 or rejected >= 100:
                        raise RuntimeError("Quá nhiều lần thử lỗi: dừng riêng run này, tiếp tục setup khác.")
                    continue
                # OOM/lỗi ở optimizer.step không được retry: trọng số có thể đã cập nhật một phần.
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
                step += 1
                consecutive_bad = 0
                losses.append(loss_sum)
                progress.update(1)
                if step % 10 == 0:
                    print(f"step={step} loss={loss_sum:.4f} rejected={rejected}", flush=True)
                if step == 10 or step % 50 == 0 or step == cfg["max_steps"]:
                    torch.cuda.synchronize()
                    stats = dict(trainable_params=count, peak_vram_gb=torch.cuda.max_memory_allocated()/1024**3,
                                 train_minutes=(time.perf_counter()-start)/60, steps=step, losses=losses,
                                 rejected_updates=rejected)
                    saved = checkpoint_safely(model, folder, cfg, stats,
                                             attempts=3 if step == cfg["max_steps"] else 1)
                    if step == cfg["max_steps"] and not saved:
                        raise RuntimeError("Train xong nhưng không lưu được checkpoint cuối; không xuất evidence giả.")
        return stats
    finally:
        model = optimizer = scheduler = scaler = loader = iterator = batch = loss = None
        params = []
        gc.collect()
        torch.cuda.empty_cache()

def load_checkpoint(folder):
    folder = Path(folder)
    run = json.loads((folder / "run.json").read_text(encoding="utf-8"))
    checkpoint = folder / run["checkpoint"] if "checkpoint" in run else folder
    model = PeftModel.from_pretrained(load_base_model(), checkpoint, is_trainable=False)
    if run["config"]["train_projector"]:
        extra = torch.load(checkpoint / "extra_trainable.pt", map_location="cpu", weights_only=True)
        named = dict(model.named_parameters())
        expected = {n for n in named if is_projector(n)}
        if set(extra) != expected:
            raise ValueError("Projector checkpoint không khớp model")
        with torch.no_grad():
            for name, weight in extra.items():
                if named[name].shape != weight.shape:
                    raise ValueError(f"Sai shape projector: {name}")
                named[name].copy_(weight.to(device=named[name].device, dtype=named[name].dtype))
    return model, run

## 7. Huấn luyện, suy luận và chấm E1–E3
Mỗi setup bắt đầu từ model gốc. Chấm val để chọn setup cho E4; test để báo cáo.
`REUSE_COMPLETED=True` bỏ qua run đã đủ steps với cùng config và dataset version.
Để debug nhanh, sửa đồng đều `MAX_STEPS=2`; kết quả đó **không phải** run nộp bài 300 steps.

In [ ]:
MAX_STEPS = 300
BATCH_SIZE, GRAD_ACCUM = 1, 8
REUSE_COMPLETED = True
dataset_version = json.loads((DATA_ROOT / "splits/meta.json").read_text(encoding="utf-8"))["dataset_version"]

def run_experiment(cfg, rows):
    cfg = dict(cfg, dataset_version=dataset_version, label_policy="skip-zero-area-v2")
    folder = RUN_ROOT / cfg["name"].lower()
    if not (folder / "run.json").exists() and DRIVE_RUN_ROOT is not None:
        backup = DRIVE_RUN_ROOT / folder.name
        if (backup / "run.json").exists():
            safe_task(f"Khôi phục {cfg['name']} từ Drive",
                      lambda: shutil.copytree(backup, folder, dirs_exist_ok=True))
    run_path = folder / "run.json"
    old = None
    try:
        old = json.loads(run_path.read_text(encoding="utf-8")) if run_path.exists() else None
    except (OSError, ValueError) as exc:
        alert("Metadata checkpoint cũ không đọc được; train lại", run=cfg["name"], error=str(exc))
    complete = old and old.get("config") == cfg and old.get("steps") == cfg["max_steps"]
    if not (REUSE_COMPLETED and complete):
        train_run(cfg, rows, folder)
    model = None
    try:
        model, run = load_checkpoint(folder)
        meta = {k: run[k] for k in ("trainable_params", "peak_vram_gb", "train_minutes")}
        meta["rejected_updates"] = run.get("rejected_updates", 0)
        results = {}
        for split in ("val", "test"):
            tag = cfg["name"].lower() + ("_val" if split == "val" else "")
            prediction = ROOT / "work" / f"{tag}.jsonl"
            predicted = safe_task(f"Inference {tag}", lambda: predict(model, splits[split], prediction))
            results[split] = (safe_task(f"Chấm {tag}", lambda: score_run(prediction, cfg["name"],
                              split=split, meta=meta, evidence_name=tag)) if predicted is not None else None)
        return results
    finally:
        del model
        gc.collect()
        torch.cuda.empty_cache()

configs, results = {}, {}
for number in (1, 2, 3):
    cfg = yaml.safe_load((ROOT / f"configs/e{number}.yaml").read_text(encoding="utf-8"))
    cfg.update(max_steps=MAX_STEPS, batch_size=BATCH_SIZE, grad_accum=GRAD_ACCUM)
    configs[cfg["name"]] = cfg
    results[cfg["name"]] = safe_task(cfg["name"], lambda: run_experiment(cfg, train_samples))

if all(results.get(n) and results[n].get("test") for n in ("E1", "E2")):
    if results["E2"]["test"]["meta"]["trainable_params"] <= results["E1"]["test"]["meta"]["trainable_params"] + 100_000_000:
        alert("Số tham số E2 khác kỳ vọng; kiểm tra projector trước khi kết luận")
for name, result in results.items():
    if result and result.get("test"):
        r = result["test"]
        print(name, "F1=", r["micro"]["f1"], "CI95=", r["f1_ci95"], "meta=", r["meta"])
    else:
        alert("Run chưa có kết quả test hợp lệ", run=name)

## 8. E4: ablation kích thước dữ liệu (10% / 100%)
Chọn setup tốt nhất **theo val**, giữ nguyên rank/lr/steps/prompt, chỉ đổi lượng dữ liệu.
Mốc 100% dùng lại run tương ứng; 10% là tập con cố định theo seed. Vẫn giữ 300 steps,
nên so sánh đo lượng dữ liệu với ngân sách optimizer cố định, không cố định số epoch.

In [ ]:
import copy

def run_e4():
    eligible = {name: r for name, r in results.items()
                if name in ("E1", "E2", "E3") and r and r.get("val") and r.get("test")}
    if not eligible:
        alert("Bỏ qua E4: chưa có E1–E3 đủ val/test để chọn baseline")
        return
    best_name = max(eligible, key=lambda name: eligible[name]["val"]["micro"]["f1"])
    best_cfg = copy.deepcopy(configs[best_name])
    print("Best theo val:", best_name)
    baseline = copy.deepcopy(results[best_name]["test"])
    baseline.update(name="E4_data_size_1.0")
    baseline["meta"].update(ablation="data_size", train_fraction=1.0, base_setup=best_name)
    out = ROOT / "report/evidence/e4_data_size_1.0.json"
    out.write_text(json.dumps(baseline, indent=2), encoding="utf-8")
    subset = random.Random(best_cfg["seed"]).sample(train_samples, max(1, round(len(train_samples)*0.1)))
    e4_cfg = copy.deepcopy(best_cfg)
    e4_cfg.update(name="E4_data_size_0.1", train_fraction=0.1, base_setup=best_name)
    results["E4_data_size_0.1"] = safe_task("E4 10%", lambda: run_experiment(e4_cfg, subset))
    print("100%:", baseline["micro"], "CI95:", baseline["f1_ci95"])
    if results["E4_data_size_0.1"] and results["E4_data_size_0.1"].get("test"):
        print("10%:", results["E4_data_size_0.1"]["test"]["micro"], "CI95:", results["E4_data_size_0.1"]["test"]["f1_ci95"])

safe_task("E4 ablation", run_e4)

### E5 (tuỳ chọn): reward IoU và self-check
Phần bắt buộc dừng ở E4. Hàm dưới thực hiện reward theo GUIDE; **chưa chạy GRPO**,
không sinh evidence E5. Muốn lấy bonus cần thêm GRPOTrainer và chạy/chấm GPU thật.

In [ ]:
from scripts.score import parse_output, iou_xyxy

def iou_reward(completion, gt_boxes):
    pred = parse_output(completion)
    if pred is None:
        return 0.0
    if not pred and not gt_boxes:
        return 1.0
    candidates = sorted(((iou_xyxy(g["bbox"], p["bbox"]), i, j)
                         for i, g in enumerate(gt_boxes) for j, p in enumerate(pred)
                         if g["label"] == p["label"]), reverse=True)
    used_gt, used_pred, total = set(), set(), 0.0
    for overlap, i, j in candidates:
        if overlap >= 0.5 and i not in used_gt and j not in used_pred:
            used_gt.add(i); used_pred.add(j); total += overlap
    return 0.2 + 0.8*total/max(len(gt_boxes), len(pred), 1)

perfect = '[{"label":"pothole","bbox_2d":[100,100,200,200]}]'
gt = parse_output(perfect)
assert iou_reward(perfect, gt) == 1.0
assert iou_reward("broken JSON", gt) == 0.0
assert iou_reward(perfect.replace("pothole", "crack"), gt) == 0.2
assert abs(iou_reward(json.dumps(json.loads(perfect)*2), gt) - 0.6) < 1e-9
assert iou_reward("[]", []) == 1.0
assert iou_reward("[]", gt) == 0.2
print("Reward self-checks passed")

### Tổng hợp và nộp bài
Điền `REPORT.md` dựa trên evidence thật và output đã xem; bổ sung ảnh minh họa, commit theo yêu cầu.
Không chạy `scripts/make_notebook.py`: script của đề sẽ ghi đè notebook đã hoàn thiện bằng khung trống.
Tham khảo API: [Qwen3-VL](https://huggingface.co/docs/transformers/model_doc/qwen3_vl),
[PEFT quantization](https://huggingface.co/docs/peft/developer_guides/quantization).

In [ ]:
for path in sorted((ROOT / "report/evidence").glob("*.json")):
    try:
        r = json.loads(path.read_text(encoding="utf-8"))
        print(f"{path.name:28s} F1={r['micro']['f1']:.4f} CI95={r['f1_ci95']} valid={r['parse_valid_rate']:.3f}")
    except (OSError, ValueError, KeyError, TypeError) as exc:
        alert("Không tổng hợp được evidence", path=str(path), error=str(exc))
print("Log cảnh báo:", ROOT / "work/alerts.jsonl")
CHECK_SUBMISSION = False
if CHECK_SUBMISSION:
    result = subprocess.run([sys.executable, "scripts/check_submission.py"])
    if result.returncode:
        alert("Bài nộp chưa đủ yêu cầu; xem danh sách FAIL phía trên")